# 📖 Notebook 3: Search Ranking & Relevance

Finding businesses near you is only half the problem. The other half is **ranking** them so the best results appear first. When you search "pizza" on Yelp, hundreds of places might match — but you want the delicious, nearby, highly-rated ones at the top.

This notebook explores how Elasticsearch scores and ranks search results by combining **text relevance**, **distance**, and **business quality** signals.

## Learning Objectives

By the end of this notebook, you'll understand:
- How Elasticsearch scores text relevance using TF-IDF / BM25
- How to boost results by rating, review count, and distance
- How to build a multi-signal ranking function
- How full-text search with fuzzy matching handles typos
- How to use Elasticsearch `function_score` for custom ranking

## 🛠️ Setup

```bash
cd system-designs/yelp
docker-compose up -d
```

**Important**: Run Notebook 1 first to index businesses into Elasticsearch, or run the setup cell below to re-index.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "yelp_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}
ES_URL = "http://localhost:9200"
INDEX_NAME = "businesses"

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def get_es():
    return Elasticsearch(ES_URL)

es = get_es()
r = get_redis()

# Test connections
try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis: {e}")

try:
    info = es.info()
    print("✅ Elasticsearch connected")
except Exception as e:
    print(f"❌ Elasticsearch: {e}")

✅ PostgreSQL connected
✅ Redis connected
✅ Elasticsearch connected


In [2]:
# Ensure the ES index exists (re-index if needed)

if not es.indices.exists(index=INDEX_NAME):
    print("Index not found — creating and loading data...")

    es.indices.create(index=INDEX_NAME, body={
        "mappings": {
            "properties": {
                "name":        {"type": "text", "analyzer": "standard"},
                "description": {"type": "text"},
                "city":        {"type": "keyword"},
                "category":    {"type": "keyword"},
                "location":    {"type": "geo_point"},
                "avg_rating":  {"type": "float"},
                "num_reviews": {"type": "integer"},
                "price_range": {"type": "integer"}
            }
        }
    })

    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT b.id, b.name, b.description, b.city, b.latitude, b.longitude,
               b.avg_rating, b.num_reviews, b.price_range, c.name AS category
        FROM businesses b JOIN categories c ON b.category_id = c.id
    """)
    businesses = cur.fetchall()
    conn.close()

    actions = [{
        "_index": INDEX_NAME, "_id": biz["id"],
        "_source": {
            "name": biz["name"], "description": biz["description"],
            "city": biz["city"], "category": biz["category"],
            "location": {"lat": float(biz["latitude"]), "lon": float(biz["longitude"])},
            "avg_rating": float(biz["avg_rating"] or 0),
            "num_reviews": biz["num_reviews"] or 0,
            "price_range": biz["price_range"]
        }
    } for biz in businesses]

    success, _ = bulk(es, actions)
    es.indices.refresh(index=INDEX_NAME)
    print(f"✅ Indexed {success} businesses")
else:
    count = es.count(index=INDEX_NAME)["count"]
    print(f"✅ ES index exists with {count} businesses")

✅ ES index exists with 500 businesses


## 🔤 Full-Text Search: How Elasticsearch Scores Text

When you search for "restaurant," Elasticsearch uses **BM25** (an improved version of TF-IDF) to score how relevant each document is:

- **TF (Term Frequency)**: How often does "restaurant" appear in this business name? More = more relevant.
- **IDF (Inverse Document Frequency)**: How rare is "restaurant" across all businesses? Rarer terms get higher scores.
- **Field length**: Shorter fields get a boost — a business named "Restaurant" scores higher than "The Best Downtown Restaurant And Bar Serving Food".

Let's see BM25 scoring in action.

In [3]:
# Basic text search — Elasticsearch assigns a relevance score to each result

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "match": {
                "name": "restaurant"
            }
        },
        "size": 10
    }
)

print(f"🔍 Search: 'restaurant'")
print(f"   Total matches: {result['hits']['total']['value']}\n")
print(f"   {'Score':>8}  {'Rating':>6}  {'Reviews':>7}  Name")
print(f"   {'-'*60}")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    print(f"   {hit['_score']:>8.3f}  ⭐{src['avg_rating']:>4.1f}  {src['num_reviews']:>7}  {src['name']}")

print(f"\n💡 Notice: results are sorted by text relevance (BM25 score), not by rating!")
print(f"   A highly-rated business might be buried if its name doesn't match well.")

🔍 Search: 'restaurant'
   Total matches: 50

      Score  Rating  Reviews  Name
   ------------------------------------------------------------
      2.355  ⭐ 3.4        8  Royal Park Restaurant
      2.355  ⭐ 3.1       15  Urban Phoenix Restaurant
      2.355  ⭐ 3.1       13  Valley Kitchen Restaurant
      2.355  ⭐ 3.2        4  Pacific Ridge Restaurant
      2.355  ⭐ 3.2       12  Blue Lane Restaurant
      2.355  ⭐ 2.6       13  Green Court Restaurant
      2.355  ⭐ 3.3        9  Mountain Plaza Restaurant
      2.355  ⭐ 3.0        8  City Park Restaurant
      2.355  ⭐ 2.7       10  Atlantic Place Restaurant
      2.355  ⭐ 3.3        7  Red Meadow Restaurant

💡 Notice: results are sorted by text relevance (BM25 score), not by rating!
   A highly-rated business might be buried if its name doesn't match well.


## 🔍 Fuzzy Matching: Handling Typos

Users make typos: "resturant", "caffee", "fitnes". Elasticsearch's fuzzy matching uses **edit distance** (Levenshtein distance) to find matches within 1-2 character changes.

In [4]:
# Search with a typo — without fuzzy matching
result_strict = es.search(
    index=INDEX_NAME,
    body={"query": {"match": {"name": "resturant"}}, "size": 5}
)

# Search with fuzzy matching enabled
result_fuzzy = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "match": {
                "name": {
                    "query": "resturant",
                    "fuzziness": "AUTO"  # allows 1-2 character edits based on word length
                }
            }
        },
        "size": 5
    }
)

print(f"🔍 Search for 'resturant' (a common typo)\n")
print(f"   Without fuzzy: {result_strict['hits']['total']['value']} results")
print(f"   With fuzzy:    {result_fuzzy['hits']['total']['value']} results\n")

if result_fuzzy["hits"]["hits"]:
    print(f"   Top fuzzy matches:")
    for hit in result_fuzzy["hits"]["hits"][:5]:
        print(f"   📍 score={hit['_score']:.3f} | {hit['_source']['name']}")

print(f"\n💡 Fuzzy matching catches typos so users still find what they want.")

🔍 Search for 'resturant' (a common typo)

   Without fuzzy: 0 results
   With fuzzy:    50 results

   Top fuzzy matches:
   📍 score=2.093 | Royal Park Restaurant
   📍 score=2.093 | Urban Phoenix Restaurant
   📍 score=2.093 | Valley Kitchen Restaurant
   📍 score=2.093 | Pacific Ridge Restaurant
   📍 score=2.093 | Blue Lane Restaurant

💡 Fuzzy matching catches typos so users still find what they want.


## 🏆 Custom Ranking with function_score

Real search ranking isn't just text relevance. Yelp combines multiple signals:

| Signal | Why It Matters |
|--------|---------------|
| **Text match** | The name/description should match the query |
| **Distance** | Closer businesses are more useful |
| **Average rating** | Higher-rated businesses should rank higher |
| **Review count** | More reviews = more trustworthy rating |

Elasticsearch's `function_score` query lets us combine all of these into a single ranking formula.

In [5]:
# Multi-signal ranking: combine text relevance + rating + distance

user_lat, user_lon = 40.758, -73.985  # Manhattan

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "function_score": {
                "query": {
                    "bool": {
                        "must": [
                            {"match": {"name": {"query": "restaurant", "fuzziness": "AUTO"}}}
                        ],
                        "filter": [
                            {"geo_distance": {"distance": "10km", "location": {"lat": user_lat, "lon": user_lon}}}
                        ]
                    }
                },
                "functions": [
                    {
                        # Boost by average rating (0-5 scale → 1-2x multiplier)
                        "field_value_factor": {
                            "field": "avg_rating",
                            "factor": 0.2,       # how strongly rating affects score
                            "modifier": "log1p",  # smooth the curve
                            "missing": 1          # default if no rating
                        },
                        "weight": 3
                    },
                    {
                        # Boost by number of reviews (more reviews = more trustworthy)
                        "field_value_factor": {
                            "field": "num_reviews",
                            "factor": 0.1,
                            "modifier": "log1p",
                            "missing": 1
                        },
                        "weight": 1
                    },
                    {
                        # Decay score based on distance (closer = higher score)
                        "gauss": {
                            "location": {
                                "origin": {"lat": user_lat, "lon": user_lon},
                                "scale": "2km",   # score drops to 50% at 2km
                                "offset": "500m",  # no decay within 500m
                                "decay": 0.5
                            }
                        },
                        "weight": 2
                    }
                ],
                "score_mode": "sum",      # combine function scores by adding them
                "boost_mode": "multiply"  # multiply function scores with query score
            }
        },
        "size": 10
    }
)

print(f"🏆 Multi-Signal Ranking: 'restaurant' near Manhattan")
print(f"   Ranking formula: text_score × (rating_boost + review_boost + distance_decay)\n")
print(f"   {'Score':>8}  {'Rating':>6}  {'Reviews':>7}  {'City':<15}  Name")
print(f"   {'-'*70}")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    print(f"   {hit['_score']:>8.2f}  ⭐{src['avg_rating']:>4.1f}  {src['num_reviews']:>7}  {src['city']:<15}  {src['name']}")

print(f"\n💡 Now the ranking considers rating, review count, AND distance — not just text match!")

🏆 Multi-Signal Ranking: 'restaurant' near Manhattan
   Ranking formula: text_score × (rating_boost + review_boost + distance_decay)

      Score  Rating  Reviews  City             Name
   ----------------------------------------------------------------------
       7.20  ⭐ 3.1       17  New York         Grand Point Restaurant
       6.73  ⭐ 3.0       12  New York         Happy Terrace Restaurant
       6.15  ⭐ 3.3        9  New York         Mountain Plaza Restaurant
       5.82  ⭐ 2.6        9  New York         Grand Kitchen Restaurant
       5.39  ⭐ 3.1       15  New York         Bay Terrace Restaurant
       4.92  ⭐ 3.6        8  New York         Central Meadow Restaurant
       4.89  ⭐ 2.9        9  New York         Downtown Ridge Restaurant
       4.74  ⭐ 2.7        9  New York         Atlantic Harbor Restaurant
       4.67  ⭐ 0.0        0  New York         Golden Dragon Restaurant
       4.64  ⭐ 3.4        8  New York         Royal Park Restaurant

💡 Now the ranking considers rati

## 📊 Comparing Ranking Strategies

Let's compare three ranking strategies side by side to see how they produce different results.

In [6]:
def search_text_only(query, city=None, size=5):
    """Rank by text relevance only."""
    body = {"query": {"match": {"name": query}}, "size": size}
    if city:
        body["query"] = {"bool": {"must": [{"match": {"name": query}}], "filter": [{"term": {"city": city}}]}}
    return es.search(index=INDEX_NAME, body=body)

def search_rating_sorted(query, city=None, size=5):
    """Match by text, sort by rating."""
    body = {"query": {"match": {"name": query}}, "sort": [{"avg_rating": "desc"}], "size": size}
    if city:
        body["query"] = {"bool": {"must": [{"match": {"name": query}}], "filter": [{"term": {"city": city}}]}}
    return es.search(index=INDEX_NAME, body=body)

def search_multi_signal(query, lat, lon, size=5):
    """Combine text relevance + rating + distance."""
    return es.search(index=INDEX_NAME, body={
        "query": {
            "function_score": {
                "query": {"bool": {
                    "must": [{"match": {"name": {"query": query, "fuzziness": "AUTO"}}}],
                    "filter": [{"geo_distance": {"distance": "10km", "location": {"lat": lat, "lon": lon}}}]
                }},
                "functions": [
                    {"field_value_factor": {"field": "avg_rating", "factor": 0.2, "modifier": "log1p", "missing": 1}, "weight": 3},
                    {"field_value_factor": {"field": "num_reviews", "factor": 0.1, "modifier": "log1p", "missing": 1}, "weight": 1},
                    {"gauss": {"location": {"origin": {"lat": lat, "lon": lon}, "scale": "2km", "offset": "500m", "decay": 0.5}}, "weight": 2}
                ],
                "score_mode": "sum", "boost_mode": "multiply"
            }
        },
        "size": size
    })


def show_results(label, result):
    print(f"\n{label}")
    for i, hit in enumerate(result["hits"]["hits"], 1):
        src = hit["_source"]
        score = hit['_score'] if hit['_score'] else 0
        print(f"   {i}. score={score:>6.2f} | ⭐{src['avg_rating']:>4.1f} ({src['num_reviews']:>2} reviews) | {src['city']:<15} | {src['name']}")


query = "cafe"
print(f"🔍 Searching for '{query}' — three different ranking strategies:\n")

show_results("📝 Strategy 1: Text Relevance Only (BM25)", search_text_only(query))
show_results("⭐ Strategy 2: Sort by Rating", search_rating_sorted(query))
show_results("🏆 Strategy 3: Multi-Signal (text + rating + distance)", search_multi_signal(query, 40.758, -73.985))

print(f"\n💡 Multi-signal ranking balances relevance, quality, and proximity.")
print(f"   This is what real search engines like Yelp actually do!")

🔍 Searching for 'cafe' — three different ranking strategies:


📝 Strategy 1: Text Relevance Only (BM25)
   1. score=  2.35 | ⭐ 3.3 ( 7 reviews) | San Francisco   | Lucky Springs Cafe
   2. score=  2.35 | ⭐ 2.8 ( 9 reviews) | San Francisco   | City Park Cafe
   3. score=  2.35 | ⭐ 3.0 (15 reviews) | San Francisco   | Fresh Park Cafe
   4. score=  2.35 | ⭐ 3.1 ( 7 reviews) | San Francisco   | Valley Lane Cafe
   5. score=  2.35 | ⭐ 3.0 ( 9 reviews) | San Francisco   | Lucky Lane Cafe

⭐ Strategy 2: Sort by Rating
   1. score=  0.00 | ⭐ 4.3 ( 3 reviews) | San Francisco   | Blue Bridge Cafe
   2. score=  0.00 | ⭐ 3.5 (11 reviews) | San Francisco   | Bay Kitchen Cafe
   3. score=  0.00 | ⭐ 3.5 (12 reviews) | San Francisco   | Valley House Cafe
   4. score=  0.00 | ⭐ 3.4 ( 9 reviews) | San Francisco   | Pacific Point Cafe
   5. score=  0.00 | ⭐ 3.4 ( 9 reviews) | San Francisco   | Blue Plaza Cafe

🏆 Strategy 3: Multi-Signal (text + rating + distance)

💡 Multi-signal ranking balances relevanc

## 🏙️ Search by City Name (Named Locations)

Users often search by city name ("pizza in San Francisco") rather than lat/lon. Yelp maps location names to polygons.

In our simplified version, we use the `city` field as a keyword filter.

In [7]:
# Search: "fitness" in Chicago

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "bool": {
                "must": [{"match": {"name": {"query": "fitness", "fuzziness": "AUTO"}}}],
                "filter": [{"term": {"city": "Chicago"}}]
            }
        },
        "sort": [{"avg_rating": "desc"}],
        "size": 10
    }
)

print(f"🏙️ 'fitness' in Chicago")
print(f"   Found {result['hits']['total']['value']} results\n")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    price = '$' * (src.get('price_range') or 1)
    print(f"   ⭐ {src['avg_rating']:.1f} ({src['num_reviews']} reviews) | {price} | {src['name']}")

print(f"\n💡 In production, Yelp maps 'Chicago' to a polygon and uses geo_shape queries.")
print(f"   For neighborhoods ('The Mission', 'SoHo'), they pre-compute which businesses")
print(f"   are in each area and store location tags on each business document.")

🏙️ 'fitness' in Chicago
   Found 0 results


💡 In production, Yelp maps 'Chicago' to a polygon and uses geo_shape queries.
   For neighborhoods ('The Mission', 'SoHo'), they pre-compute which businesses
   are in each area and store location tags on each business document.


## ⌨️ Autocomplete (Search-As-You-Type)

When you start typing `"piz"` into the Yelp search box, suggestions appear **before you finish the word**. This is called **autocomplete** (or *search-as-you-type*) and it's one of the first things users interact with.

The trick: instead of full-word matching, we match on **prefixes**. Elasticsearch has a few ways to do this; the simplest is the `match_phrase_prefix` query, which treats the **last token** as a prefix.

**Why not just `LIKE 'piz%'` in SQL?**

- Autocomplete needs to be *fast* (< 50 ms) because it fires on every keystroke.
- `LIKE 'piz%'` works with a B-tree, but only on the leading prefix of the full field — not on *words inside* the name.
- `"The Best Pizza"` would not match `LIKE 'piz%'` (starts with "The"), but users expect it to.
- Elasticsearch indexes each **word** independently, so "Pizza" is findable no matter where it sits in the name.

For production at Yelp's scale you'd use dedicated features like `search_as_you_type` fields or the *completion suggester* (which builds a special in-memory data structure). We'll stick with `match_phrase_prefix` here — it's zero-config and good enough to see the idea.


In [8]:
# Autocomplete: suggest businesses as the user types

def autocomplete(prefix, limit=5):
    """Return top business names that match a partial query like 'piz' or 'coffe'."""
    result = es.search(index=INDEX_NAME, body={
        "query": {
            "match_phrase_prefix": {
                "name": {"query": prefix, "max_expansions": 20}
            }
        },
        # Tiebreak by how well-known the business is (more reviews = more likely what the user meant)
        "sort": ["_score", {"num_reviews": "desc"}],
        "size": limit,
        # We only need the name for the dropdown — tell ES to skip everything else
        "_source": ["name", "city", "avg_rating", "num_reviews"],
    })
    return [hit["_source"] for hit in result["hits"]["hits"]]


# Simulate a user typing "res" → "rest" → "resta" one letter at a time
typed = ""
for ch in "resta":
    typed += ch
    start = time.time()
    suggestions = autocomplete(typed)
    elapsed_ms = (time.time() - start) * 1000
    print(f"⌨️  User typed '{typed}' → {elapsed_ms:>5.1f} ms → {len(suggestions)} suggestions")
    for s in suggestions[:3]:
        print(f"       • {s['name']} ({s['city']}) — ⭐{s['avg_rating']:.1f}, {s['num_reviews']} reviews")
    print()

print("💡 Each keystroke fires a fresh query, but each one is still fast because")
print("   Elasticsearch indexed every word in every business name.")


⌨️  User typed 'r' →   3.5 ms → 5 suggestions
       • Red Ridge Restaurant (New York) — ⭐3.0, 2 reviews
       • Red Ridge Spa (Chicago) — ⭐2.6, 11 reviews
       • Royal Ridge Bar & Grill (Los Angeles) — ⭐0.0, 0 reviews

⌨️  User typed 're' →   3.0 ms → 5 suggestions
       • Red Meadow Restaurant (New York) — ⭐3.3, 7 reviews
       • Red Ridge Restaurant (New York) — ⭐3.0, 2 reviews
       • Red Kitchen Restaurant (New York) — ⭐0.0, 0 reviews

⌨️  User typed 'res' →   2.4 ms → 5 suggestions
       • Grand Point Restaurant (New York) — ⭐3.1, 17 reviews
       • Urban Phoenix Restaurant (New York) — ⭐3.1, 15 reviews
       • Bay Terrace Restaurant (New York) — ⭐3.1, 15 reviews

⌨️  User typed 'rest' →   2.3 ms → 5 suggestions
       • Grand Point Restaurant (New York) — ⭐3.1, 17 reviews
       • Urban Phoenix Restaurant (New York) — ⭐3.1, 15 reviews
       • Bay Terrace Restaurant (New York) — ⭐3.1, 15 reviews

⌨️  User typed 'resta' →   2.1 ms → 5 suggestions
       • Grand Point Res

## ⚡ Caching Popular Search Results

Popular searches ("restaurants in New York", "coffee in San Francisco") are repeated thousands of times per minute. We cache them in Redis.

In [9]:
def ranked_search_cached(query, lat, lon, radius_km=10, category=None, limit=10):
    """
    Full-featured search with multi-signal ranking and Redis caching.
    This is what a real Yelp search API endpoint might look like.
    """
    # Build cache key from search parameters
    cache_key = f"ranked:{query}:{round(lat,2)}:{round(lon,2)}:{radius_km}:{category or 'all'}:{limit}"

    # Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True

    # Build ES query
    must_clauses = [{"match": {"name": {"query": query, "fuzziness": "AUTO"}}}]
    if category:
        must_clauses.append({"term": {"category": category}})

    result = es.search(index=INDEX_NAME, body={
        "query": {
            "function_score": {
                "query": {"bool": {
                    "must": must_clauses,
                    "filter": [{"geo_distance": {"distance": f"{radius_km}km", "location": {"lat": lat, "lon": lon}}}]
                }},
                "functions": [
                    {"field_value_factor": {"field": "avg_rating", "factor": 0.2, "modifier": "log1p", "missing": 1}, "weight": 3},
                    {"field_value_factor": {"field": "num_reviews", "factor": 0.1, "modifier": "log1p", "missing": 1}, "weight": 1},
                    {"gauss": {"location": {"origin": {"lat": lat, "lon": lon}, "scale": "2km", "offset": "500m", "decay": 0.5}}, "weight": 2}
                ],
                "score_mode": "sum", "boost_mode": "multiply"
            }
        },
        "size": limit
    })

    # Format results
    results = []
    for hit in result["hits"]["hits"]:
        src = hit["_source"]
        results.append({
            "name": src["name"], "city": src["city"], "category": src["category"],
            "avg_rating": src["avg_rating"], "num_reviews": src["num_reviews"],
            "price_range": src.get("price_range", 1),
            "score": round(hit["_score"], 2)
        })

    # Cache for 30 seconds
    r.setex(cache_key, 30, json.dumps(results))
    return results, False


# Benchmark: cold vs cached search
r.flushdb()  # clear cache

start = time.time()
results, from_cache = ranked_search_cached("restaurant", 40.758, -73.985, radius_km=5)
t1 = (time.time() - start) * 1000
print(f"🔍 Cold search: {t1:.2f} ms (cached: {from_cache})")

start = time.time()
results, from_cache = ranked_search_cached("restaurant", 40.758, -73.985, radius_km=5)
t2 = (time.time() - start) * 1000
print(f"⚡ Cached search: {t2:.2f} ms (cached: {from_cache})")
print(f"\n🚀 Cache speedup: {t1/t2:.1f}×\n")

print(f"Top results:")
for i, biz in enumerate(results[:5], 1):
    price = '$' * biz['price_range']
    print(f"   {i}. score={biz['score']:>6.2f} | ⭐{biz['avg_rating']:>4.1f} ({biz['num_reviews']} reviews) | {price:<4} | {biz['name']}")

🔍 Cold search: 6.00 ms (cached: False)
⚡ Cached search: 0.35 ms (cached: True)

🚀 Cache speedup: 17.3×

Top results:
   1. score=  7.20 | ⭐ 3.1 (17 reviews) | $    | Grand Point Restaurant
   2. score=  6.73 | ⭐ 3.0 (12 reviews) | $$   | Happy Terrace Restaurant
   3. score=  6.15 | ⭐ 3.3 (9 reviews) | $    | Mountain Plaza Restaurant
   4. score=  5.82 | ⭐ 2.6 (9 reviews) | $$   | Grand Kitchen Restaurant
   5. score=  5.39 | ⭐ 3.1 (15 reviews) | $$$  | Bay Terrace Restaurant


## 🧹 Cleanup

In [10]:
# Clean up Redis cache and Elasticsearch index
r = get_redis()
keys = r.keys("ranked:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned {len(keys)} Redis cache keys")

es = get_es()
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"🧹 Deleted ES index '{INDEX_NAME}'")

print("🧹 Cleanup complete!")

🧹 Cleaned 1 Redis cache keys


🧹 Deleted ES index 'businesses'
🧹 Cleanup complete!


## 📚 Summary

### Key Takeaways

1. **BM25 text scoring** gives a baseline relevance score — but it's not enough alone
2. **Fuzzy matching** catches typos using edit distance (Levenshtein)
3. **Multi-signal ranking** combines text relevance, rating, review count, and distance
4. **`function_score`** in Elasticsearch lets you build custom ranking formulas
5. **Redis caching** makes popular searches near-instant (sub-millisecond)
6. **Named locations** (cities, neighborhoods) are mapped to polygons for area-based search

### System Design Interview Tips

- **Elasticsearch vs Postgres**: Use ES for full-text search at scale, Postgres with `pg_trgm` for simpler setups
- **Ranking signals**: Always mention multiple ranking factors — text match alone is insufficient
- **Data sync**: If using ES alongside a primary DB, you need **Change Data Capture (CDC)** to keep them in sync
- **Filter sequence**: Apply the most restrictive filter first (usually distance) to shrink the search space
- **Keep it simple**: At Yelp's scale (10M businesses, ~10GB), even Postgres can work. Don't over-engineer!

### Architecture Summary

```
User Search Request
        │
        ▼
   ┌─────────┐     cache hit     ┌────────┐
   │  Redis   │◄─────────────────│ API GW │
   │  Cache   │                  └────┬───┘
   └─────────┘                        │ cache miss
                                      ▼
                              ┌───────────────┐
                              │ Elasticsearch  │  ← geo_distance + text match + function_score
                              └───────┬───────┘
                                      │ CDC sync
                                      ▼
                              ┌───────────────┐
                              │  PostgreSQL    │  ← source of truth for businesses & reviews
                              └───────────────┘
```